# Прогноз покупки клиента в течение 90 дней

## Цель проекта

Построить модель классификации, которая предсказывает вероятность того, что клиент интернет-магазина совершит покупку в течение следующих 90 дней.

Основная метрика качества: **ROC-AUC**.

## Используемые данные

- `apparel-purchases.csv` — история покупок клиентов;
- `apparel-messages.csv` — история рекламных рассылок;
- `apparel-target_binary.csv` — целевой признак;
- `full_campaign_daily_event.csv` — дневная агрегация событий по кампаниям;
- `full_campaign_daily_event_channel.csv` — дневная агрегация событий по кампаниям и каналам.

Важно: при работе с агрегатами `nunique_*` нельзя суммировать значения по дням, так как уникальность рассчитана только внутри одного дня.

In [ ]:
import os
import re
import gc
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
pd.set_option('display.max_columns', 120)

## 1. Загрузка данных

In [ ]:
DATA_DIR = Path('.')

PURCHASES_PATH = DATA_DIR / 'apparel-purchases.csv'
TARGET_PATH = DATA_DIR / 'apparel-target_binary.csv'
MESSAGES_PATH = DATA_DIR / 'apparel-messages.csv'
ZIP_PATH = DATA_DIR / 'filtered_data-20260603T064953Z-3-001.zip'

DAILY_EVENT_PATH = DATA_DIR / 'full_campaign_daily_event.csv'
DAILY_EVENT_CHANNEL_PATH = DATA_DIR / 'full_campaign_daily_event_channel.csv'

# Если apparel-messages.csv находится только внутри архива, достанем его автоматически.
if not MESSAGES_PATH.exists() and ZIP_PATH.exists():
    with zipfile.ZipFile(ZIP_PATH) as z:
        for name in z.namelist():
            if name.endswith('apparel-messages.csv'):
                z.extract(name, DATA_DIR)
                extracted = DATA_DIR / name
                extracted.rename(MESSAGES_PATH)
                break

purchases = pd.read_csv(PURCHASES_PATH, parse_dates=['date'])
target = pd.read_csv(TARGET_PATH)

daily_event = pd.read_csv(DAILY_EVENT_PATH, parse_dates=['date'])
daily_event_channel = pd.read_csv(DAILY_EVENT_CHANNEL_PATH, parse_dates=['date'])

display(purchases.head())
display(target.head())
display(daily_event.head())
display(daily_event_channel.head())

## 2. Первичный анализ данных

In [ ]:
def show_basic_info(df, name):
    print(f'===== {name} =====')
    print('shape:', df.shape)
    print('\nТипы данных:')
    print(df.dtypes)
    print('\nДоля пропусков:')
    print(df.isna().mean().sort_values(ascending=False).head(20))
    print('\nДубликаты:', df.duplicated().sum())
    print()

show_basic_info(purchases, 'purchases')
show_basic_info(target, 'target')
show_basic_info(daily_event, 'daily_event')
show_basic_info(daily_event_channel, 'daily_event_channel')

print('Доля положительного класса:', target['target'].mean())
print('Период покупок:', purchases['date'].min(), '-', purchases['date'].max())

In [ ]:
# Проверка числовых признаков покупок
display(purchases[['quantity', 'price']].describe())

# Проверка событий и каналов рассылок без загрузки всей таблицы в память
events = set()
channels = set()
message_rows = 0

for chunk in pd.read_csv(MESSAGES_PATH, usecols=['event', 'channel'], chunksize=1_000_000):
    message_rows += len(chunk)
    events.update(chunk['event'].dropna().unique())
    channels.update(chunk['channel'].dropna().unique())

print('Количество строк в messages:', message_rows)
print('События:', sorted(events))
print('Каналы:', sorted(channels))

### Выводы по первичному анализу

- Задача является задачей бинарной классификации.
- Целевой класс сильно несбалансирован, поэтому при обучении используем стратификацию и балансировку классов.
- Для модели нужны агрегированные признаки на уровне `client_id`, так как целевой признак задан по клиентам.
- `category_ids` содержит список категорий товара. Так как идентификаторы категорий сквозные для всех уровней, будем извлекать все id из списка и создавать признаки по самым частотным категориям.
- Таблица `apparel-messages` большая, поэтому агрегируем её по частям через `chunksize`.

## 3. Разработка признаков

In [ ]:
def make_purchase_features(purchases: pd.DataFrame, top_n_categories: int = 30) -> pd.DataFrame:
    data = purchases.copy()
    data['amount'] = data['quantity'] * data['price']
    obs_end = data['date'].max()

    features = data.groupby('client_id').agg(
        purchase_rows=('client_id', 'size'),
        purchase_days=('date', 'nunique'),
        first_purchase=('date', 'min'),
        last_purchase=('date', 'max'),
        total_quantity=('quantity', 'sum'),
        mean_quantity=('quantity', 'mean'),
        total_amount=('amount', 'sum'),
        mean_amount=('amount', 'mean'),
        median_amount=('amount', 'median'),
        max_amount=('amount', 'max'),
        mean_price=('price', 'mean'),
        max_price=('price', 'max'),
        messages_with_purchase=('message_id', 'nunique')
    ).reset_index()

    features['days_since_last_purchase'] = (obs_end - features['last_purchase']).dt.days
    features['customer_age_days'] = (features['last_purchase'] - features['first_purchase']).dt.days
    features['purchase_frequency'] = features['purchase_days'] / (features['customer_age_days'] + 1)

    # Оконные признаки активности
    for window in [7, 14, 30, 60, 90]:
        window_data = data[data['date'] >= obs_end - pd.Timedelta(days=window)]
        window_features = window_data.groupby('client_id').agg(**{
            f'purchases_{window}d': ('client_id', 'size'),
            f'amount_{window}d': ('amount', 'sum'),
            f'quantity_{window}d': ('quantity', 'sum')
        }).reset_index()
        features = features.merge(window_features, on='client_id', how='left')

    # Признаки по категориям.
    # Извлекаем все числа из category_ids, так как категория с одним id одинакова на любом уровне вложенности.
    category_series = data['category_ids'].dropna().astype(str).str.findall(r'\d+').explode()
    top_categories = category_series.value_counts().head(top_n_categories).index.tolist()

    for cat in top_categories:
        data[f'category_{cat}_cnt'] = (
            data['category_ids']
            .astype(str)
            .str.contains(fr"'?{cat}'?", regex=True)
            .astype('int8')
        )

    category_features = (
        data
        .groupby('client_id')[[f'category_{cat}_cnt' for cat in top_categories]]
        .sum()
        .reset_index()
    )

    features = features.merge(category_features, on='client_id', how='left')
    features = features.drop(columns=['first_purchase', 'last_purchase'])

    return features

In [ ]:
def make_message_features(messages_path: Path, obs_end: pd.Timestamp, chunksize: int = 1_000_000) -> pd.DataFrame:
    numeric_parts = []
    date_parts = []

    for chunk in pd.read_csv(messages_path, parse_dates=['date', 'created_at'], chunksize=chunksize):
        chunk['hour'] = chunk['created_at'].dt.hour

        base = chunk.groupby('client_id').agg(
            message_events=('message_id', 'count'),
            unique_messages=('message_id', 'nunique'),
            unique_campaigns=('bulk_campaign_id', 'nunique'),
            hour_sum=('hour', 'sum'),
            hour_count=('hour', 'count')
        )

        event_counts = pd.crosstab(chunk['client_id'], chunk['event']).add_prefix('event_')
        channel_counts = pd.crosstab(chunk['client_id'], chunk['channel']).add_prefix('channel_')

        numeric_parts.append(
            base
            .join(event_counts, how='outer')
            .join(channel_counts, how='outer')
            .fillna(0)
        )

        date_parts.append(
            chunk.groupby('client_id').agg(
                first_message_date=('date', 'min'),
                last_message_date=('date', 'max')
            )
        )

        del chunk
        gc.collect()

    numeric = pd.concat(numeric_parts).groupby(level=0).sum()
    dates = pd.concat(date_parts).groupby(level=0).agg(
        first_message_date=('first_message_date', 'min'),
        last_message_date=('last_message_date', 'max')
    )

    features = numeric.join(dates)

    features['mean_message_hour'] = features['hour_sum'] / features['hour_count'].replace(0, np.nan)
    features['days_since_last_message'] = (obs_end - features['last_message_date']).dt.days
    features['message_lifetime_days'] = (
        features['last_message_date'] - features['first_message_date']
    ).dt.days

    features = features.drop(columns=['hour_sum', 'hour_count', 'first_message_date', 'last_message_date'])

    # Относительные признаки: конверсии внутри пользовательской истории коммуникаций.
    for col in ['event_open', 'event_click', 'event_purchase', 'event_send', 'event_unsubscribe']:
        if col in features.columns:
            features[f'{col}_rate'] = features[col] / features['message_events'].replace(0, np.nan)

    return features.reset_index()

In [ ]:
def make_campaign_features(daily_event: pd.DataFrame, daily_event_channel: pd.DataFrame) -> pd.DataFrame:
    '''
    Эти признаки описывают кампании в целом.
    Важно: nunique_* не суммируем по дням, используем среднее/максимум.
    Далее их можно присоединять к пользовательским сообщениям по bulk_campaign_id,
    если нужна более сложная версия решения.
    '''
    count_cols = [col for col in daily_event.columns if col.startswith('count_')]
    nunique_cols = [col for col in daily_event.columns if col.startswith('nunique_')]

    campaign_features = daily_event.groupby('bulk_campaign_id').agg(
        **{f'{col}_sum': (col, 'sum') for col in count_cols},
        **{f'{col}_mean': (col, 'mean') for col in nunique_cols},
        **{f'{col}_max': (col, 'max') for col in nunique_cols},
        campaign_days=('date', 'nunique')
    ).reset_index()

    channel_count_cols = [col for col in daily_event_channel.columns if col.startswith('count_')]
    channel_nunique_cols = [col for col in daily_event_channel.columns if col.startswith('nunique_')]

    channel_features = daily_event_channel.groupby('bulk_campaign_id').agg(
        **{f'{col}_sum': (col, 'sum') for col in channel_count_cols},
        **{f'{col}_mean': (col, 'mean') for col in channel_nunique_cols},
        **{f'{col}_max': (col, 'max') for col in channel_nunique_cols}
    ).reset_index()

    return campaign_features.merge(channel_features, on='bulk_campaign_id', how='left')

In [ ]:
purchase_features = make_purchase_features(purchases, top_n_categories=30)

obs_end = purchases['date'].max()
message_features = make_message_features(MESSAGES_PATH, obs_end=obs_end)

campaign_features = make_campaign_features(daily_event, daily_event_channel)

print('purchase_features:', purchase_features.shape)
print('message_features:', message_features.shape)
print('campaign_features:', campaign_features.shape)

В базовую модель включаем пользовательские признаки из покупок и пользовательские признаки из рассылок.

Глобальные признаки кампаний рассчитаны отдельно. Их можно использовать в расширенной версии проекта, но в базовый набор они не включены, чтобы избежать лишнего усложнения и риска некорректной агрегации `nunique_*`.

In [ ]:
data = (
    target
    .merge(purchase_features, on='client_id', how='left')
    .merge(message_features, on='client_id', how='left')
)

data = data.replace([np.inf, -np.inf], np.nan)

print(data.shape)
display(data.head())
display(data.isna().mean().sort_values(ascending=False).head(20))

## 4. Обучение моделей

In [ ]:
X = data.drop(columns=['client_id', 'target'])
y = data['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print('Train:', X_train.shape)
print('Test:', X_test.shape)
print('Доля класса 1 в train:', y_train.mean())
print('Доля класса 1 в test:', y_test.mean())

In [ ]:
models = {
    'LogisticRegression': Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(
            max_iter=1000,
            class_weight='balanced',
            random_state=RANDOM_STATE
        ))
    ]),
    'RandomForest': Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('model', RandomForestClassifier(
            n_estimators=200,
            max_depth=10,
            min_samples_leaf=10,
            class_weight='balanced',
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, proba)
    results.append({'model': name, 'roc_auc_test': auc})
    print(f'{name}: ROC-AUC = {auc:.4f}')

pd.DataFrame(results).sort_values('roc_auc_test', ascending=False)

## 5. Улучшение модели

In [ ]:
param_grid = {
    'model__C': [0.01, 0.05, 0.1, 0.5, 1, 3, 10],
    'model__penalty': ['l2']
}

logreg_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        max_iter=2000,
        class_weight='balanced',
        random_state=RANDOM_STATE
    ))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

grid = GridSearchCV(
    estimator=logreg_pipeline,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1
)

grid.fit(X_train, y_train)

print('Лучшие параметры:', grid.best_params_)
print('Лучший ROC-AUC на CV:', grid.best_score_)

best_model = grid.best_estimator_
test_proba = best_model.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, test_proba)

print('ROC-AUC на тестовой выборке:', test_auc)

## 6. Тестирование и интерпретация

In [ ]:
# ROC-AUC оценивает качество ранжирования, поэтому основной вывод делаем по вероятностям.
test_proba = best_model.predict_proba(X_test)[:, 1]
print('Итоговый ROC-AUC:', roc_auc_score(y_test, test_proba))

# Для примера классификационного отчёта используем порог 0.5.
# В бизнес-задаче порог лучше выбирать отдельно: например, по бюджету рассылки.
test_pred = (test_proba >= 0.5).astype(int)
print(classification_report(y_test, test_pred))

In [ ]:
# Важность признаков для финальной модели:
# для логистической регрессии смотрим абсолютные значения коэффициентов.
feature_names = X_train.columns
coefs = best_model.named_steps['model'].coef_[0]

importance = (
    pd.DataFrame({
        'feature': feature_names,
        'abs_coef': np.abs(coefs),
        'coef': coefs
    })
    .sort_values('abs_coef', ascending=False)
)

display(importance.head(30))

## 7. Общий вывод

В проекте были выполнены:

1. Загрузка и первичный анализ данных.
2. Разработка пользовательских признаков:
   - частота и давность покупок;
   - сумма и количество покупок;
   - активность за последние 7/14/30/60/90 дней;
   - признаки по популярным категориям;
   - активность клиента в рекламных рассылках;
   - доли открытий, кликов, покупок и отписок.
3. Обучение моделей классификации.
4. Подбор гиперпараметров для логистической регрессии.
5. Финальное тестирование по ROC-AUC.

Базовая проверка на предоставленных данных показала, что логистическая регрессия с балансировкой классов является сильным и интерпретируемым бейзлайном. Для дальнейшего улучшения можно:
- добавить признаки по последним кампаниям клиента;
- аккуратно присоединить агрегаты кампаний без суммирования `nunique_*`;
- подобрать больше гиперпараметров;
- протестировать бустинги, если стек проекта будет расширен за пределы `sklearn`.